# RG3 F1 목표: 조건부 순위/상태 부스팅 앙상블

93번과 고정 RG3 RH620개/7개 구간 동일. CatBoost(depth3/150), LightGBM(leaves7/depth3/150), XGBoost(depth2/150) 고정 평균 전문가를 순위75/절대상태10에 각각 추가. 외부 평가로 early stopping/튜닝하지 않음. 후보 전문가5개의 단독 및 두 전문가 .5 혼합15개, 내부 OOF F1 최적 임계값 선택. 기존 단독 RF는 비교용, 각 선정 전문가는 여러 모델 평균이다.

평가 구간은 피처변환/정상 이웃/분류/앙상블/임계값 학습에서 제외. 원본 포함 정상601·가스19, CN7 미사용. 알려진 자료의 개발용 중첩 검증이며 새 독립 검증 아니다. primary 행은 inner_F1_selected_blend. 모델 캐시와 ref/그룹/시각/변환센서 저장: 후속 실험에서도 평가 구간이 저장된 ref에 포함되면 안 된다. F1 .5 보장 없음.


In [1]:
from pathlib import Path
import json,numpy as np,pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import RobustScaler
from sklearn.neighbors import NearestNeighbors
from sklearn.ensemble import RandomForestClassifier,ExtraTreesClassifier
from sklearn.metrics import confusion_matrix,average_precision_score,precision_recall_curve
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
root=Path.cwd()
if not (root/'data').exists():root=root.parent
tables=root/'reports/tables'
s=''.join(json.loads((root/'notebooks/67_rg3_recall_first_nested_mode_ensemble.ipynb').read_text(encoding='utf-8'))['cells'][1]['source'])
exec(s.split('F=pd.DataFrame')[0])
assert len(d)==620 and y.sum()==19 and (y==0).sum()==601
stdcols=[x+'__past10_std' for x in ['Barrel_Temperature_2','Hopper_Temperature','Cushion_Position','Barrel_Temperature_5']]
state=d[['Hopper_Temperature','Barrel_Temperature_2','Barrel_Temperature_5','Cushion_Position','Plasticizing_Time']].copy()
state['pressure_switch_gap']=d.Max_Injection_Pressure-d.Max_Switch_Over_Pressure
memo={};audit=[];feature_cache={}
def predict(ref,q):
 assert set(groups.iloc[ref]).isdisjoint(groups.iloc[q])
 key=tuple(ref)
 if key not in memo:
  normal=ref[y[ref]==0];all_ids=np.arange(len(d))
  im=SimpleImputer(strategy='median',keep_empty_features=True).fit(state.iloc[normal])
  raw=im.transform(state)
  si=SimpleImputer(strategy='median',keep_empty_features=True).fit(d.iloc[ref][stdcols])
  std=si.transform(d[stdcols]);base=np.c_[raw,std]
  normal_ids=normal;norm=base[normal]
  scaler=RobustScaler().fit(norm);context=np.clip(scaler.transform(base),-8,8)
  rank=np.empty_like(base);k=min(75,len(normal)-1)
  for col in range(base.shape[1]):
   other=np.arange(base.shape[1])!=col
   nn=NearestNeighbors(n_neighbors=k+1).fit(context[normal][:,other])
   indices=nn.kneighbors(context[:,other],return_distance=False)
   for row in all_ids:
    ids=indices[row][normal_ids[indices[row]]!=row][:k]
    peers=norm[ids,col]
    rank[row,col]=(1+(peers<base[row,col]).sum()+.5*(peers==base[row,col]).sum())/(len(peers)+2)
  features=[std,rank,base];scores=[];original=None
  for view,F in enumerate(features):
   predictions=[]
   for cls in [RandomForestClassifier,ExtraTreesClassifier]:
    for seed in [42,7]:
     m=cls(n_estimators=200 if view==0 else 80,max_depth=8 if view==0 else 4,min_samples_leaf=3 if view==0 else 4,class_weight='balanced_subsample' if view==0 else 'balanced',random_state=seed,n_jobs=2)
     m.fit(F[ref],y[ref]);p=m.predict_proba(F)[:,1]
     predictions.append(p)
     if view==0 and cls==RandomForestClassifier and seed==42:original=p
   scores.append(np.mean(predictions,axis=0))
  boost_scores=[]
  for F in [rank,base]:
   boosting=[]
   models=[
    CatBoostClassifier(iterations=150,depth=3,learning_rate=.05,l2_leaf_reg=10,auto_class_weights='Balanced',random_seed=42,thread_count=2,verbose=False,allow_writing_files=False),
    LGBMClassifier(n_estimators=150,num_leaves=7,max_depth=3,min_child_samples=15,learning_rate=.03,reg_lambda=10,class_weight='balanced',random_state=42,n_jobs=2,verbosity=-1),
    XGBClassifier(n_estimators=150,max_depth=2,min_child_weight=5,learning_rate=.03,reg_lambda=10,subsample=.8,colsample_bytree=.8,scale_pos_weight=float((y[ref]==0).sum()/y[ref].sum()),random_state=42,n_jobs=2,eval_metric='logloss')]
   for model in models:
    model.fit(F[ref],y[ref]);boosting.append(model.predict_proba(F)[:,1])
   boost_scores.append(np.mean(boosting,axis=0))
  memo[key]=np.column_stack(scores+[original]+boost_scores)
  feature_cache[key]=(std,rank,base)
  audit.append(dict(train=len(ref),gas_training=int(y[ref].sum()),normal_training=len(normal),train_blocks=','.join(map(str,groups.iloc[ref].unique())),excluded_blocks=','.join(map(str,groups[~groups.index.isin(ref)].unique()))))
 return memo[key][q]
def stats(yy,p):
 tn,fp,fn,tp=confusion_matrix(yy,p,labels=[0,1]).ravel()
 return dict(TP=int(tp),FP=int(fp),FN=int(fn),TN=int(tn),F1=2*tp/(2*tp+fp+fn) if 2*tp+fp+fn else 0,recall=tp/(tp+fn) if tp+fn else 0)
def choose_cut(yy,score):
 precision,recall,cuts=precision_recall_curve(yy,score)
 f1=np.divide(2*precision[:-1]*recall[:-1],precision[:-1]+recall[:-1],out=np.zeros_like(cuts),where=(precision[:-1]+recall[:-1])>0)
 order=np.lexsort((cuts,recall[:-1],f1));cut=float(cuts[order[-1]])
 return cut,stats(yy,score>=cut)
# Predeclared simplex weights; pure experts already contain four models.
expert_columns=[0,1,2,4,5]
weights=[np.eye(5)[i] for i in range(5)]+[(np.eye(5)[i]+np.eye(5)[j])/2 for i in range(5) for j in range(i+1,5)]
records=[];choices=[];search=[]
for block in blocks:
 q=np.flatnonzero(groups.eq(block));ref=np.flatnonzero(~groups.eq(block))
 out=predict(ref,q);inner=np.full((len(ref),6),np.nan)
 for ib in groups.iloc[ref].unique():
  mask=groups.iloc[ref].eq(ib).to_numpy();inner[mask]=predict(ref[~mask],ref[mask])
 assert np.isfinite(inner).all()
 def record(name,pred,score,cut):
  for j,i in enumerate(q):records.append(dict(model=name,TimeStamp=d.TimeStamp.iloc[i],block=int(block),actual=int(y[i]),prediction=int(pred[j]),score=float(score[j]),threshold=float(cut)))
 candidates=[]
 for w in weights:
  score=inner[:,expert_columns]@w;cut,result=choose_cut(y[ref],score);o=out[:,expert_columns]@w
  candidates.append((result['F1'],result['recall'],-result['FP'],w,cut,o))
  search.append(dict(block=int(block),std_weight=w[0],rank_weight=w[1],state_weight=w[2],rank_boost_weight=w[3],state_boost_weight=w[4],threshold=cut,**result))
 chosen=max(candidates,key=lambda z:z[:3])
 record('inner_F1_selected_blend',chosen[5]>=chosen[4],chosen[5],chosen[4])
 choices.append(dict(block=int(block),std_weight=chosen[3][0],rank_weight=chosen[3][1],state_weight=chosen[3][2],rank_boost_weight=chosen[3][3],state_boost_weight=chosen[3][4],threshold=chosen[4],inner_F1=chosen[0],inner_recall=chosen[1]))
 for col,name in enumerate(['deep_std4_ensemble','rank75_ensemble','absolute_state10_ensemble','original_RF_configuration','rank75_boost_ensemble','absolute_state10_boost_ensemble']):
  cut,result=choose_cut(y[ref],inner[:,col]);o=out[:,col]
  record(name+'_inner_F1_cut',o>=cut,o,cut)
  cut=float(np.quantile(inner[y[ref]==0,col],.99,method='higher'))
  record(name+'_normal99_cut',o>cut,o,cut)
 print('Finished RG3 F1-selection block',block,flush=True)
p=pd.DataFrame(records);rows=[]
for name,g in p.groupby('model'):
 row=stats(g.actual,g.prediction);rows.append(dict(model=name,AP=average_precision_score(g.actual,g.score),**row))
m=pd.DataFrame(rows)
for obj,name in [(p,'predictions'),(m,'metrics'),(pd.DataFrame(choices),'inner_choices'),(pd.DataFrame(search),'inner_search'),(pd.DataFrame(audit),'training_audit')]:obj.to_csv(tables/f'rg3_F1_boost_selection_{name}.csv',index=False,encoding='utf-8-sig')
print(m.to_string(index=False))
print('Frozen 620-row RG3-only development cohort; nested block evaluation, not fresh independent validation.')


# Preserve fold-local fitting references and transformed sensor features for auditable follow-up.
cache={'labels':y,'groups':groups.to_numpy(),'timestamps':d.TimeStamp.astype(str).to_numpy(dtype='U')}
for j,(ref,score) in enumerate(memo.items()):
 cache[f'ref_{j}']=np.array(ref,int);cache[f'scores_{j}']=score
 for v,F in enumerate(feature_cache[ref]):cache[f'features_{j}_{v}']=F
np.savez_compressed(tables/'rg3_F1_boost_fold_cache.npz',**cache)
print('Cached',len(memo),'training reference sets; stored fit references must be honored in follow-ups.')


Finished RG3 F1-selection block 1


Finished RG3 F1-selection block 2


Finished RG3 F1-selection block 3


Finished RG3 F1-selection block 4


Finished RG3 F1-selection block 5


Finished RG3 F1-selection block 6


Finished RG3 F1-selection block 7


                                       model       AP  TP  FP  FN  TN       F1   recall
absolute_state10_boost_ensemble_inner_F1_cut 0.063166   4  53  15 548 0.105263 0.210526
absolute_state10_boost_ensemble_normal99_cut 0.063166   1  12  18 589 0.062500 0.052632
      absolute_state10_ensemble_inner_F1_cut 0.072270   5 210  14 391 0.042735 0.263158
      absolute_state10_ensemble_normal99_cut 0.072270   1  14  18 587 0.058824 0.052632
             deep_std4_ensemble_inner_F1_cut 0.079209   4  50  15 551 0.109589 0.210526
             deep_std4_ensemble_normal99_cut 0.079209   1  11  18 590 0.064516 0.052632
                     inner_F1_selected_blend 0.069945   3  19  16 582 0.146341 0.157895
      original_RF_configuration_inner_F1_cut 0.080563   6  53  13 548 0.153846 0.315789
      original_RF_configuration_normal99_cut 0.080563   1  13  18 588 0.060606 0.052632
          rank75_boost_ensemble_inner_F1_cut 0.071127   2  24  17 577 0.088889 0.105263
          rank75_boost_ensemble_

Cached 28 training reference sets; stored fit references must be honored in follow-ups.
